# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 05: Multilingual Processing (Detection, Translation & Normalization)

**Goal**: Process customer complaints and audio transcripts across multiple languages (English, Bengali, Hindi, Tamil, Telugu, etc.) using `facebook/nllb-200-distilled-600M`:
1. **Language Detection**: Automatically recognize language and map between ISO-639-1, Whisper tokens, and NLLB BCP-47 codes.
2. **Preservation**: Keep original customer text and transcripts intact.
3. **Normalized Translation**: Translate non-English complaint evidence into English for cross-lingual downstream modeling.
4. **Persistent Caching**: Cache all translations to eliminate duplicate model inferences.
5. **Memory Efficiency**: Lazily load NLLB model and release GPU memory upon completion.

In [ ]:
# Environment & Module Path Setup
import os
import sys
import json
from pathlib import Path

CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

from src.multilingual.language_detector import LanguageDetector
from src.multilingual.translator import NLLBTranslator

detector = LanguageDetector()
print('Multilingual modules loaded successfully!')

### 1. Language Detection Across Diverse Scripts
Test the language detector on sample customer complaint texts in English, Bengali, Hindi, Tamil, and Telugu.

In [ ]:
test_phrases = [
    ('English', 'The plastic bottle has a pungent chemical smell and leaks water.'),
    ('Bengali', 'প্লাস্টিকের বোতল থেকে তীব্র রাসায়নিক গন্ধ বের হচ্ছে এবং পানি পড়ছে।'),
    ('Hindi', 'स्क्रीन काम नहीं कर रही है और बार-बार झिलमिला रही है।'),
    ('Tamil', 'இந்த தயாரிப்பு மிகவும் சேதமடைந்துள்ளது மற்றும் வேலை செய்யவில்லை.'),
    ('Telugu', 'ఈ ఉత్పత్తి సరిగ్గా పని చేయడం లేదు, నాణ్యత చాలా పేలవంగా ఉంది.')
]

print('Language Detection Results:')
for label, phrase in test_phrases:
    det = detector.detect_language(phrase)
    print(f"{label:10s} -> ISO: {det['iso_code']:3s} | NLLB: {det['nllb_code']:10s} | Script: {det['script']:10s} | Conf: {det['confidence']}")

### 2. Multilingual Translation with NLLB-200 & Caching
We initialize `NLLBTranslator`. When the input is already in English, it bypasses model inference automatically.

In [ ]:
translator = NLLBTranslator(
    model_id='facebook/nllb-200-distilled-600M',
    default_target_lang='eng_Latn',
    cache_path='data/processed/translation_cache.json'
)

# 1. English input (zero-cost passthrough)
en_res = translator.translate('The bottle cap is not locking properly.', src_lang='en', tgt_lang='en')
print(f"English Passthrough: {en_res['translated_text']} (Cached: {en_res['cached']})")

# 2. Benchmark Bengali translation check
bn_text = 'প্লাস্টিকের বোতল থেকে তীব্র রাসায়নিক গন্ধ বের হচ্ছে।'
bn_res = translator.translate(bn_text, src_lang='bn', tgt_lang='en')
print(f"\nBengali Input : {bn_text}")
print(f"English Output: {bn_res['translated_text']} (Cached: {bn_res['cached']})")

# 3. Benchmark Hindi translation check
hi_text = 'स्क्रीन काम नहीं कर रही है और बार-बार झिलमिला रही है।'
hi_res = translator.translate(hi_text, src_lang='hi', tgt_lang='en')
print(f"\nHindi Input   : {hi_text}")
print(f"English Output: {hi_res['translated_text']} (Cached: {hi_res['cached']})")

### 3. Verify Cache Hit (Sub-Millisecond Retrieval)
Re-running the same phrase demonstrates disk-backed cache retrieval without GPU compute.

In [ ]:
import time

t0 = time.time()
cached_res = translator.translate(bn_text, src_lang='bn', tgt_lang='en')
elapsed_ms = (time.time() - t0) * 1000

print(f"Cached Retrieval Result: {cached_res['translated_text']}")
print(f"Was Cached: {cached_res['cached']} | Elapsed Time: {elapsed_ms:.2f} ms")
assert cached_res['cached'] is True, 'Expected cached result!'

### 4. GPU Memory Cleanup
In a multi-model system on Kaggle's dual T4 GPUs (16 GB per GPU), we must never leave unused models in VRAM.

In [ ]:
translator.release_memory()
print('NLLB-200 VRAM cleared. Ready for Aspect & Emotion Classifier Training!')

### Summary & Next Steps
- Multi-script language detection and code mapping are active.
- NLLB-200 translation with caching and memory release is verified.
- Proceed to **Notebook 06: Aspect & Emotion Classifier Training**.